In [1]:
!pip install -q \
    transformers==4.40.2 \
    accelerate==0.30.1 \
    sentencepiece \
    bitsandbytes \
    pandas \
    numpy \
    tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 92.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [3]:
NBME_PATH = "/kaggle/input/nbme-score-clinical-patient-notes"

patient_notes = pd.read_csv(f"{NBME_PATH}/patient_notes.csv")
train = pd.read_csv(f"{NBME_PATH}/train.csv")

annotated_ids = set(train["pn_num"].unique())

nbme_annotated = patient_notes[
    patient_notes["pn_num"].isin(annotated_ids)
][["pn_num", "pn_history"]].reset_index(drop=True)

print("Annotated notes:", len(nbme_annotated))


Annotated notes: 1000


In [4]:
annotated_pn_nums = set(train["pn_num"].unique())

nbme_annotated = patient_notes[
    patient_notes["pn_num"].isin(annotated_pn_nums)
][["pn_num", "pn_history"]].reset_index(drop=True)

print("Annotated notes only:", len(nbme_annotated))


Annotated notes only: 1000


In [6]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN2")

login(token=hf_token)
print("HF user:", whoami())


HF user: {'type': 'user', 'id': '6975d4b70066194d08508639', 'name': 'krishnamidulakarnan', 'fullname': 'krishnamidula karnan', 'email': 'krishnakarnan2017@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1769904000, 'isPro': False, 'avatarUrl': '/avatars/7c4afcfd25f7774c73ccb466754f249a.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'kaggle-gemma-access', 'role': 'read', 'createdAt': '2026-01-27T12:59:50.642Z'}}}


In [7]:
MODEL_NAME = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Gemma's activation function should be approximate GeLU and not exact GeLU.
Changing the activation function to `gelu_pytorch_tanh`.if you want to use the legacy `gelu`, edit the `model.config` to set `hidden_activation=gelu`   instead of `hidden_act`. See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
      )
    )
    (norm): GemmaR

In [8]:
MAX_CHARS = 1500

def truncate(text):
    return text[:MAX_CHARS] if isinstance(text, str) else ""

def build_prompt(note):
    messages = [
        {
            "role": "user",
            "content": (
                "You are a clinical information extraction system.\n\n"
                "TASK:\n"
                "Extract patient-reported symptoms or clinician-observed findings.\n\n"
                "STRICT RULES:\n"
                "- Extract symptoms only\n"
                "- Do NOT include diagnoses, medications, procedures, labs\n"
                "- Do NOT include negated symptoms\n"
                "- Do NOT include instructions or rules\n"
                "- Output one symptom per line starting with '-'\n"
                "- If none, output exactly: none\n\n"
                "CLINICAL NOTE:\n"
                f"{truncate(note)}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


In [9]:
def parse_symptoms(output_text):
    if not isinstance(output_text, str):
        return []

    text = output_text.lower().strip()

    if text == "none":
        return []

    banned = (
        "extract",
        "do not",
        "rule",
        "instruction",
        "clinical note",
        "output",
    )

    symptoms = []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("-"):
            s = line[1:].strip()
            if s and not any(b in s for b in banned):
                symptoms.append(s)

    # Deduplicate
    seen = set()
    clean = []
    for s in symptoms:
        if s not in seen:
            seen.add(s)
            clean.append(s)

    return clean


In [10]:
BATCH_SIZE = 8          # Gemma-2B is lightweight
MAX_NEW_TOKENS = 64
CHECKPOINT_EVERY = 100

OUT_DIR = "/kaggle/working/nbme_gemma_best"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILE = f"{OUT_DIR}/gemma_predictions_annotated.csv"


In [11]:
processed = set()
if os.path.exists(OUT_FILE):
    df_prev = pd.read_csv(OUT_FILE)
    processed = set(df_prev["pn_num"])
    print("Resuming from", len(processed))
else:
    print("Fresh run")


Fresh run


In [12]:
results = []

for i in tqdm(range(0, len(nbme_annotated), BATCH_SIZE)):
    batch = nbme_annotated.iloc[i:i+BATCH_SIZE]
    batch = batch[~batch["pn_num"].isin(processed)]

    if batch.empty:
        continue

    prompts = [build_prompt(t) for t in batch["pn_history"]]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pn, out in zip(batch["pn_num"], decoded):
        results.append({
            "pn_num": pn,
            "model": "gemma-2b-it",
            "predicted_symptoms": parse_symptoms(out)
        })

    if len(results) >= CHECKPOINT_EVERY:
        df = pd.DataFrame(results)
        if os.path.exists(OUT_FILE):
            df.to_csv(OUT_FILE, mode="a", header=False, index=False)
        else:
            df.to_csv(OUT_FILE, index=False)

        processed.update(df["pn_num"])
        results = []
        torch.cuda.empty_cache()


  0%|          | 0/125 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [13]:
if results:
    df = pd.DataFrame(results)
    if os.path.exists(OUT_FILE):
        df.to_csv(OUT_FILE, mode="a", header=False, index=False)
    else:
        df.to_csv(OUT_FILE, index=False)

print("Gemma BEST-POSSIBLE inference complete.")


Gemma BEST-POSSIBLE inference complete.


In [15]:
df = pd.read_csv(OUT_FILE)
print("Total predictions:", len(df))
df.head()


Total predictions: 1000


,pn_num,model,predicted_symptoms
0,16,gemma-2b-it,"['palpitations', 'chest pressure', 'shortness ..."
1,41,gemma-2b-it,"['sob', 'chest pain', 'pressure on chest', 'co..."
2,46,gemma-2b-it,"['palpitations', 'nervousness', 'anxiousness',..."
3,82,gemma-2b-it,"['heart pounding', 'lightheadedness', 'shortne..."
4,100,gemma-2b-it,"['endorses light headedness, chest pressure th..."


In [21]:
NBME_PATH = "/kaggle/input/nbme-score-clinical-patient-notes"

train = pd.read_csv(f"{NBME_PATH}/train.csv")
features = pd.read_csv(f"{NBME_PATH}/features.csv")

print("Annotations:", train.shape)
print("Features:", features.shape)


Annotations: (14300, 6)
Features: (143, 3)


In [24]:
import os

os.listdir("/kaggle/working/nbme_gemma_annotated")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/nbme_gemma_annotated'

In [25]:


preds = pd.read_csv("/kaggle/working/nbme_gemma_best/gemma_predictions_annotated.csv")

print("Predictions:", preds.shape)
preds.head()


Predictions: (1000, 3)


,pn_num,model,predicted_symptoms
0,16,gemma-2b-it,"['palpitations', 'chest pressure', 'shortness ..."
1,41,gemma-2b-it,"['sob', 'chest pain', 'pressure on chest', 'co..."
2,46,gemma-2b-it,"['palpitations', 'nervousness', 'anxiousness',..."
3,82,gemma-2b-it,"['heart pounding', 'lightheadedness', 'shortne..."
4,100,gemma-2b-it,"['endorses light headedness, chest pressure th..."


In [26]:
# feature_num → feature_text
feature_map = dict(zip(features["feature_num"], features["feature_text"]))
train["feature_text"] = train["feature_num"].map(feature_map)

gt_per_note = (
    train.groupby("pn_num")["feature_text"]
    .apply(list)
    .reset_index()
)

gt_per_note.head()


,pn_num,feature_text
0,16,[Family-history-of-MI-OR-Family-history-of-myo...
1,41,[Family-history-of-MI-OR-Family-history-of-myo...
2,46,[Family-history-of-MI-OR-Family-history-of-myo...
3,82,[Family-history-of-MI-OR-Family-history-of-myo...
4,100,[Family-history-of-MI-OR-Family-history-of-myo...


In [27]:
eval_df = preds.merge(gt_per_note, on="pn_num", how="inner")

eval_df = eval_df.rename(columns={
    "predicted_symptoms": "predicted",
    "feature_text": "gold"
})

print("Evaluation notes:", len(eval_df))
eval_df.head()


Evaluation notes: 1000


,pn_num,model,predicted,gold
0,16,gemma-2b-it,"['palpitations', 'chest pressure', 'shortness ...",[Family-history-of-MI-OR-Family-history-of-myo...
1,41,gemma-2b-it,"['sob', 'chest pain', 'pressure on chest', 'co...",[Family-history-of-MI-OR-Family-history-of-myo...
2,46,gemma-2b-it,"['palpitations', 'nervousness', 'anxiousness',...",[Family-history-of-MI-OR-Family-history-of-myo...
3,82,gemma-2b-it,"['heart pounding', 'lightheadedness', 'shortne...",[Family-history-of-MI-OR-Family-history-of-myo...
4,100,gemma-2b-it,"['endorses light headedness, chest pressure th...",[Family-history-of-MI-OR-Family-history-of-myo...


In [28]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_list(items):
    return [normalize_text(x) for x in items if isinstance(x, str)]


In [29]:
def is_match(pred, gold):
    return pred == gold or pred in gold or gold in pred


In [31]:
import re

In [32]:
records = []

for _, row in eval_df.iterrows():
    pred_list = normalize_list(eval(row["predicted"]))
    gold_list = normalize_list(row["gold"])

    matched_gold = set()
    tp = 0

    for p in pred_list:
        for g in gold_list:
            if g not in matched_gold and is_match(p, g):
                matched_gold.add(g)
                tp += 1
                break

    fp = max(len(pred_list) - tp, 0)
    fn = max(len(gold_list) - tp, 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    records.append({
        "pn_num": row["pn_num"],
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

per_note_df = pd.DataFrame(records)
per_note_df.head()


,pn_num,tp,fp,fn,precision,recall,f1
0,16,2,4,11,0.333333,0.153846,0.210526
1,41,0,4,13,0.000000,0.000000,0.000000
2,46,0,5,13,0.000000,0.000000,0.000000
3,82,3,2,10,0.600000,0.230769,0.333333
4,100,1,14,12,0.066667,0.076923,0.071429


In [33]:
mean_precision = per_note_df["precision"].mean()
mean_recall = per_note_df["recall"].mean()
mean_f1 = per_note_df["f1"].mean()

print(f"Precision: {mean_precision:.4f}")
print(f"Recall:    {mean_recall:.4f}")
print(f"F1-score:  {mean_f1:.4f}")


Precision: 0.2628
Recall:    0.1164
F1-score:  0.1553


In [34]:
def bootstrap_ci(values, n_bootstrap=1000, alpha=0.05):
    rng = np.random.default_rng(42)
    samples = []
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        samples.append(sample.mean())
    lower = np.percentile(samples, 100 * (alpha / 2))
    upper = np.percentile(samples, 100 * (1 - alpha / 2))
    return lower, upper


prec_ci = bootstrap_ci(per_note_df["precision"].values)
rec_ci = bootstrap_ci(per_note_df["recall"].values)
f1_ci = bootstrap_ci(per_note_df["f1"].values)

print("Precision CI:", prec_ci)
print("Recall CI:", rec_ci)
print("F1 CI:", f1_ci)


Precision CI: (np.float64(0.2506739668460624), np.float64(0.2754683602569815))
Recall CI: (np.float64(0.11139026646556059), np.float64(0.12103787408873806))
F1 CI: (np.float64(0.1486715328924409), np.float64(0.16171110350049817))


In [37]:
import json

In [38]:
OUT_DIR = "/kaggle/working/nbme_gemma_eval"
os.makedirs(OUT_DIR, exist_ok=True)

per_note_df.to_csv(f"{OUT_DIR}/gemma_per_note_scores.csv", index=False)

metrics = {
    "dataset": "nbme",
    "model": "gemma-2b-it",
    "n_notes": len(per_note_df),
    "precision": mean_precision,
    "recall": mean_recall,
    "f1": mean_f1,
    "precision_ci": prec_ci,
    "recall_ci": rec_ci,
    "f1_ci": f1_ci
}

with open(f"{OUT_DIR}/gemma_nbme_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Gemma NBME evaluation saved.")


Gemma NBME evaluation saved.


old

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN2")

login(token=hf_token)
print("HF user:", whoami())


In [ ]:
MODEL_NAME = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()


In [ ]:
BATCH_SIZE = 8          # Gemma-2B is lightweight
MAX_NEW_TOKENS = 64
CHECKPOINT_EVERY = 100

OUTPUT_DIR = "/kaggle/working/nbme_gemma_annotated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = f"{OUTPUT_DIR}/gemma_predictions_annotated.csv"


In [ ]:
processed_ids = set()

if os.path.exists(OUTPUT_FILE):
    prev = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(prev["pn_num"].tolist())
    print("Resuming from", len(processed_ids), "notes")
else:
    print("Starting fresh inference")


In [ ]:
results = []
total_notes = len(nbme_annotated)

for start_idx in tqdm(range(0, total_notes, BATCH_SIZE)):
    batch = nbme_annotated.iloc[start_idx:start_idx + BATCH_SIZE]
    batch = batch[~batch["pn_num"].isin(processed_ids)]

    if batch.empty:
        continue

    prompts = [format_prompt(t) for t in batch["pn_history"].tolist()]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.0,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    for pn, out in zip(batch["pn_num"], decoded):
        results.append({
            "pn_num": pn,
            "model": "gemma-2b-it",
            "predicted_symptoms": parse_symptoms(out)
        })

    if len(results) >= CHECKPOINT_EVERY:
        df = pd.DataFrame(results)
        if os.path.exists(OUTPUT_FILE):
            df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)
        else:
            df.to_csv(OUTPUT_FILE, index=False)

        processed_ids.update(df["pn_num"].tolist())
        results = []
        torch.cuda.empty_cache()


In [ ]:
if results:
    df = pd.DataFrame(results)
    if os.path.exists(OUTPUT_FILE):
        df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)
    else:
        df.to_csv(OUTPUT_FILE, index=False)

print("Gemma-2B annotated NBME inference complete.")


In [ ]:
df = pd.read_csv(OUTPUT_FILE)
print("Total predictions:", len(df))
df.head()
